
# CODE REVIEW BUDDY
# A comprehensive code analysis tool using LangChain and Gemini API
# Author: Saumy DhoLu


# 1: INSTALLATION AND IMPORTS

In [9]:
# Install required packages
!pip install langchain langchain-google-genai gitpython pygments tabulate colorama -q


In [10]:
# Standard library imports
import os
import ast
import re
import sys
import json
import zipfile
import tempfile
import subprocess
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Any

# Third-party imports
import pandas as pd
from pygments import highlight
from pygments.lexers import get_lexer_by_name, guess_lexer
from pygments.formatters import TerminalFormatter
from tabulate import tabulate
from colorama import Fore, Style, init

# LangChain imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage, SystemMessage
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

# IPython display imports
from IPython.display import display, Markdown, HTML, clear_output

# Initialize colorama
init(autoreset=True)

print(" All packages installed successfully!")
print(" Code Review Buddy is ready to launch!")


 All packages installed successfully!
 Code Review Buddy is ready to launch!


# 2: CONFIGURATION AND SETUP


In [12]:
def load_api_key(filepath='GEMINI_API_KEY.txt'):
    """Load Gemini API key from local text file"""
    try:
        with open(filepath, 'r') as f:
            api_key = f.read().strip()
        
        if not api_key:
            raise ValueError("API key file is empty")
        
        return api_key
    except FileNotFoundError:
        raise FileNotFoundError(
            f"API key file '{filepath}' not found. "
            "Please create a file named 'GEMINI_API_KEY.txt' "
            "in the same directory with your Gemini API key."
        )
    except Exception as e:
        raise Exception(f"Error loading API key: {e}")

In [13]:
# Load API key
try:
    GEMINI_API_KEY = load_api_key()
    
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        google_api_key=GEMINI_API_KEY,
        temperature=0.1,
        max_output_tokens=4096,
        timeout=60
    )
    
    print("Gemini API configured successfully!")
    
except Exception as e:
    print(f"Error configuring Gemini API: {e}")
    print("Please ensure GEMINI_API_KEY.txt exists in the current directory")

Gemini API configured successfully!


In [14]:
class Config:
    """Configuration class for Code Review Buddy"""

    # Supported file extensions and their languages
    SUPPORTED_EXTENSIONS = {
        '.py': 'python',
        '.js': 'javascript',
        '.jsx': 'javascript',
        '.ts': 'typescript',
        '.java': 'java',
        '.cpp': 'cpp',
        '.c': 'c',
        '.cs': 'csharp',
        '.php': 'php',
        '.rb': 'ruby',
        '.go': 'go',
        '.rs': 'rust',
        '.ipynb': 'jupyterfile'
    }

    # File size limits (in bytes)
    MAX_FILE_SIZE = 2048 * 2048  # 2MB
    MAX_TOTAL_SIZE = 20 * 2048 * 2048  # 40MB

    # Analysis thresholds
    MAX_LINE_LENGTH = 200
    MAX_COMPLEXITY = 20
    MAX_NESTING_DEPTH = 8

In [15]:
class DetailedReportManager:
    """Manages creation and storage of detailed markdown reports"""
    
    def __init__(self, output_dir="code_review_reports"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
        
    def save_detailed_report(self, filename, analysis_type, content):
        """Save detailed analysis to a markdown file"""
        safe_filename = re.sub(r'[^\w\-_.]', '_', filename)
        report_filename = f"{safe_filename}_{analysis_type}_{self.timestamp}.md"
        report_path = self.output_dir / report_filename
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(f"# {analysis_type.title()} Analysis: {filename}\n\n")
            f.write(f"**Generated:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            f.write("---\n\n")
            f.write(content)
        
        return str(report_path)

# 3: KNOWLEDGE BASE LOADER

In [17]:
class KnowledgeBase:
    """Loads and manages the Python code review knowledge base"""
    
    def __init__(self, filepath="knowledge_base.txt"):
        self.rules = {}
        self.categories = {}
        self.filepath = filepath
        self.load_knowledge_base()
    
    def load_knowledge_base(self):
        """Load rules from an external text file instead of hardcoding"""
        
        with open(self.filepath, "r", encoding="utf-8") as f:
            knowledge_content = f.read()
        self.parse_knowledge_content(knowledge_content)
    
    def parse_knowledge_content(self, content):
        """Parse the knowledge base content and organize rules"""
        
        lines = content.strip().split('\n')
        current_rule = {}
        current_category = "general"
        
        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Detect section headers
            if line.startswith("# Section"):
                current_category = line.split(":", 1)[-1].strip()
                if current_category not in self.categories:
                    self.categories[current_category] = []
                continue

            if line.startswith("Rule:"):
                if current_rule:
                    self.add_rule(current_rule, current_category)
                current_rule = {"name": line[5:].strip(), "category": current_category}
            elif line.startswith("Description:"):
                current_rule["description"] = line[12:].strip()
            elif line.startswith("Example:") or line.startswith("Bad:") or line.startswith("Good:") or line.startswith("Exception:"):
                # Keep extra info if available
                current_rule.setdefault("extra", []).append(line)

        if current_rule:
            self.add_rule(current_rule, current_category)

    def add_rule(self, rule, category):
        """Add a rule to the knowledge base"""
        rule_name = rule["name"]
        self.rules[rule_name] = rule

        if category not in self.categories:
            self.categories[category] = []
        self.categories[category].append(rule_name)

    def get_rules_by_category(self, category):
        """Get all rules for a specific category"""
        return [self.rules[name] for name in self.categories.get(category, [])]

    def get_all_rules(self):
        """Get all rules in the knowledge base"""
        return list(self.rules.values())

# Initialize knowledge base
knowledge_base = KnowledgeBase("knowledge_base.txt")
print(f"Knowledge base loaded with {len(knowledge_base.rules)} rules")


Knowledge base loaded with 22 rules


In [18]:
# Print first few rules under Clarity
for rule in knowledge_base.get_rules_by_category("Clarity, Readability, and PEP 8")[:3]:
    print(f"{rule['name']}: {rule['description']}")
    if "extra" in rule:
        print("  Extras:", rule["extra"])

Naming Conventions.: Variable and function names should be in snake_case (e.g., `user_profile`). Class names should be in PascalCase (e.g., `UserProfile`). Constants should be in ALL_CAPS (e.g., `MAX_RETRIES`).
  Extras: ['Exception: When integrating with legacy code that uses a different style.']
Line Length.: Keep lines under 79-99 characters to ensure they are readable on a variety of screen sizes and tools.
  Extras: ['Exception: Long URLs, import statements, or string constants that are difficult to break.']
Docstrings.: Every public module, function, class, and method should have a docstring. Use a standard format like Google Style or reStructuredText to explain the purpose, arguments (`Args:`), and return values (`Returns:`).
  Extras: ['Example: """A brief summary.']


# 4: FILE HANDLING AND UPLOAD SYSTEM

In [20]:
class LocalFileHandler:
    """Handles file uploads and processing for local environment"""
    
    def __init__(self):
        self.supported_extensions = Config.SUPPORTED_EXTENSIONS
        self.processed_files = {}
    
    def get_upload_choice(self):
        """Present upload options to user"""
        print("\n" + "="*60)
        print("CODE UPLOAD OPTIONS")
        print("="*60)
        print("1. Upload Individual Files (specify paths)")
        print("2. Upload ZIP Archive")
        print("3. Clone GitHub Repository")
        print("4. Scan Directory")
        print("="*60)
        
        while True:
            try:
                choice = input("\nEnter your choice (1-4): ").strip()
                if choice in ['1', '2', '3', '4']:
                    return int(choice)
                else:
                    print("Please enter a valid choice (1-4)")
            except (ValueError, KeyboardInterrupt):
                print("Please enter a valid choice (1-4)")
    
    def upload_individual_files(self):
        """Handle individual file uploads via file paths"""
        print("\nEnter file paths (one per line, empty line to finish):")
        print("Example: /path/to/your/file.py")
        
        file_paths = []
        while True:
            path = input("File path: ").strip()
            if not path:
                break
            file_paths.append(path)
        
        if not file_paths:
            print("No files provided")
            return {}
        
        processed_files = {}
        total_size = 0
        
        for filepath in file_paths:
            filepath = os.path.expanduser(filepath)  # Expand ~ to home directory
            
            if not os.path.exists(filepath):
                print(f" Skipping {filepath}: File not found")
                continue
            
            if not os.path.isfile(filepath):
                print(f" Skipping {filepath}: Not a file")
                continue
            
            file_size = os.path.getsize(filepath)
            total_size += file_size
            
            # Check file size limits
            if file_size > Config.MAX_FILE_SIZE:
                print(f" Skipping {filepath}: File too large ({file_size/1024:.1f}KB > {Config.MAX_FILE_SIZE/1024:.0f}KB)")
                continue
            
            if total_size > Config.MAX_TOTAL_SIZE:
                print(f" Stopping upload: Total size limit exceeded")
                break
            
            # Check if file type is supported
            file_ext = Path(filepath).suffix.lower()
            if file_ext not in self.supported_extensions:
                print(f" Skipping {filepath}: Unsupported file type ({file_ext})")
                continue
            
            try:
                # Read file content
                with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
                    file_content = f.read()
                
                filename = os.path.basename(filepath)
                processed_files[filename] = {
                    'content': file_content,
                    'language': self.supported_extensions[file_ext],
                    'size': file_size,
                    'path': filename
                }
                
                print(f" Processed {filename} ({file_size/1024:.1f}KB)")
            
            except Exception as e:
                print(f" Error processing {filepath}: {e}")
        
        return processed_files
    
    def upload_zip_archive(self):
        """Handle ZIP file upload and extraction"""
        zip_path = input("\nEnter path to ZIP file: ").strip()
        zip_path = os.path.expanduser(zip_path)
        
        if not os.path.exists(zip_path):
            print(f" ZIP file not found: {zip_path}")
            return {}
        
        if not zip_path.lower().endswith('.zip'):
            print(f" Not a ZIP file: {zip_path}")
            return {}
        
        processed_files = {}
        
        try:
            # Create temporary directory for extraction
            with tempfile.TemporaryDirectory() as temp_dir:
                # Extract ZIP file
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(temp_dir)
                
                # Process extracted files
                extracted_files = self.scan_directory(temp_dir)
                processed_files.update(extracted_files)
        
        except Exception as e:
            print(f"Error processing ZIP file: {e}")
        
        return processed_files
    
    def clone_github_repo(self):
        """Handle GitHub repository cloning"""
        print("\nEnter GitHub repository URL:")
        repo_url = input("Repository URL: ").strip()
        
        if not repo_url:
            print("No repository URL provided")
            return {}
        
        # Validate GitHub URL format
        if not re.match(r'https://github\.com/[\w\-\.]+/[\w\-\.]+', repo_url):
            print("Invalid GitHub URL format")
            return {}
        
        try:
            # Create temporary directory for cloning
            with tempfile.TemporaryDirectory() as temp_dir:
                clone_dir = os.path.join(temp_dir, 'repo')
                
                print(f" Cloning repository...")
                result = subprocess.run(
                    ['git', 'clone', '--depth', '1', repo_url, clone_dir],
                    capture_output=True,
                    text=True,
                    timeout=30
                )
                
                if result.returncode != 0:
                    print(f"Git clone failed: {result.stderr}")
                    return {}
                
                print(" Repository cloned successfully")
                
                # Process cloned files
                processed_files = self.scan_directory(clone_dir)
                return processed_files
        
        except subprocess.TimeoutExpired:
            print(" Repository cloning timed out")
            return {}
        except Exception as e:
            print(f" Error cloning repository: {e}")
            return {}
    
    def scan_directory(self, directory=None):
        """Recursively scan directory for supported code files"""
        if directory is None:
            directory = input("\nEnter directory path to scan: ").strip()
            directory = os.path.expanduser(directory)
        
        if not os.path.exists(directory):
            print(f" Directory not found: {directory}")
            return {}
        
        if not os.path.isdir(directory):
            print(f" Not a directory: {directory}")
            return {}
        
        processed_files = {}
        total_size = 0
        
        for root, dirs, files in os.walk(directory):
            # Skip hidden directories and common ignore patterns
            dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ['node_modules', '__pycache__', '.git', 'venv', 'env']]
            
            for file in files:
                file_path = os.path.join(root, file)
                relative_path = os.path.relpath(file_path, directory)
                
                # Skip hidden files
                if file.startswith('.'):
                    continue
                
                file_ext = Path(file).suffix.lower()
                if file_ext not in self.supported_extensions:
                    continue
                
                try:
                    file_size = os.path.getsize(file_path)
                    total_size += file_size
                    
                    # Check size limits
                    if file_size > Config.MAX_FILE_SIZE:
                        print(f"Skipping {relative_path}: File too large")
                        continue
                    
                    if total_size > Config.MAX_TOTAL_SIZE:
                        print(f"Stopping scan: Total size limit exceeded")
                        break
                    
                    # Read file content
                    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
                        content = f.read()
                    
                    processed_files[relative_path] = {
                        'content': content,
                        'language': self.supported_extensions[file_ext],
                        'size': file_size,
                        'path': relative_path
                    }
                    
                    print(f"Found {relative_path} ({file_size/1024:.1f}KB)")
                
                except Exception as e:
                    print(f"Error reading {relative_path}: {e}")
        
        return processed_files
    
    def process_files(self):
        """Main method to handle file processing based on user choice"""
        choice = self.get_upload_choice()
        
        if choice == 1:
            return self.upload_individual_files()
        elif choice == 2:
            return self.upload_zip_archive()
        elif choice == 3:
            return self.clone_github_repo()
        elif choice == 4:
            return self.scan_directory()
        
        return {}


# 5: CODE ANALYSIS ENGINE

In [22]:
class CodeAnalyzer:
    """Core code analysis engine with multiple analysis methods"""

    def __init__(self):
        self.metrics = {}
        self.issues = []
        self.suggestions = []

    def analyze_python_ast(self, code_content, filename):
        """Analyze Python code using AST parsing"""
        try:
            tree = ast.parse(code_content)

            analysis = {
                'complexity': self.calculate_cyclomatic_complexity(tree),
                'structure': self.analyze_code_structure(tree),
                'imports': self.analyze_imports(tree),
                'functions': self.analyze_functions(tree),
                'classes': self.analyze_classes(tree),
                'issues': []
            }

            # Check for specific issues
            analysis['issues'].extend(self.check_mutable_defaults(tree))
            analysis['issues'].extend(self.check_bare_except(tree))
            analysis['issues'].extend(self.check_eval_usage(tree))

            return analysis

        except SyntaxError as e:
            return {
                'error': f"Syntax Error at line {e.lineno}: {e.msg}",
                'issues': [{'type': 'syntax', 'line': e.lineno, 'message': e.msg}]
            }
        except Exception as e:
            return {
                'error': f"Analysis Error: {str(e)}",
                'issues': []
            }

    def calculate_cyclomatic_complexity(self, tree):
        """Calculate cyclomatic complexity of the code"""
        complexity = 1  # Base complexity

        for node in ast.walk(tree):
            if isinstance(node, (ast.If, ast.While, ast.For, ast.With)):
                complexity += 1
            elif isinstance(node, ast.Try):
                complexity += len(node.handlers)
            elif isinstance(node, ast.BoolOp):
                complexity += len(node.values) - 1

        return complexity

    def analyze_code_structure(self, tree):
        """Analyze overall code structure"""
        structure = {
            'total_lines': 0,
            'blank_lines': 0,
            'comment_lines': 0,
            'code_lines': 0,
            'functions': 0,
            'classes': 0,
            'imports': 0,
            'max_nesting_depth': 0
        }

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                structure['functions'] += 1
            elif isinstance(node, ast.ClassDef):
                structure['classes'] += 1
            elif isinstance(node, (ast.Import, ast.ImportFrom)):
                structure['imports'] += 1

        return structure

    def analyze_imports(self, tree):
        """Analyze import statements"""
        imports = {
            'stdlib': [],
            'third_party': [],
            'local': [],
            'issues': []
        }

        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    imports['third_party'].append(alias.name)
            elif isinstance(node, ast.ImportFrom):
                module = node.module or ''
                if module.startswith('.'):
                    imports['local'].append(module)
                else:
                    imports['third_party'].append(module)

        return imports

    def analyze_functions(self, tree):
        """Analyze function definitions"""
        functions = []

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                func_info = {
                    'name': node.name,
                    'line': node.lineno,
                    'args_count': len(node.args.args),
                    'has_docstring': ast.get_docstring(node) is not None,
                    'complexity': self.calculate_function_complexity(node),
                    'issues': []
                }

                # Check for mutable default arguments
                for default in node.args.defaults:
                    if isinstance(default, (ast.List, ast.Dict, ast.Set)):
                        func_info['issues'].append({
                            'type': 'mutable_default',
                            'message': 'Mutable default argument detected'
                        })

                functions.append(func_info)

        return functions

    def analyze_classes(self, tree):
        """Analyze class definitions"""
        classes = []

        for node in ast.walk(tree):
            if isinstance(node, ast.ClassDef):
                class_info = {
                    'name': node.name,
                    'line': node.lineno,
                    'methods': [],
                    'has_docstring': ast.get_docstring(node) is not None,
                    'base_classes': len(node.bases)
                }

                # Analyze methods
                for item in node.body:
                    if isinstance(item, ast.FunctionDef):
                        class_info['methods'].append({
                            'name': item.name,
                            'line': item.lineno,
                            'has_docstring': ast.get_docstring(item) is not None
                        })

                classes.append(class_info)

        return classes

    def calculate_function_complexity(self, func_node):
        """Calculate complexity for a specific function"""
        complexity = 1

        for node in ast.walk(func_node):
            if isinstance(node, (ast.If, ast.While, ast.For, ast.With)):
                complexity += 1
            elif isinstance(node, ast.Try):
                complexity += len(node.handlers)

        return complexity

    def check_mutable_defaults(self, tree):
        """Check for mutable default arguments"""
        issues = []

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                for i, default in enumerate(node.args.defaults):
                    if isinstance(default, (ast.List, ast.Dict, ast.Set)):
                        issues.append({
                            'type': 'mutable_default',
                            'line': node.lineno,
                            'message': f"Function '{node.name}' has mutable default argument",
                            'severity': 'high',
                            'category': 'correctness'
                        })

        return issues

    def check_bare_except(self, tree):
        """Check for bare except clauses"""
        issues = []

        for node in ast.walk(tree):
            if isinstance(node, ast.ExceptHandler):
                if node.type is None:
                    issues.append({
                        'type': 'bare_except',
                        'line': node.lineno,
                        'message': 'Bare except clause should specify exception type',
                        'severity': 'medium',
                        'category': 'correctness'
                    })

        return issues

    def check_eval_usage(self, tree):
        """Check for dangerous eval/exec usage"""
        issues = []

        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Name):
                    if node.func.id in ['eval', 'exec']:
                        issues.append({
                            'type': 'dangerous_eval',
                            'line': node.lineno,
                            'message': f"Use of '{node.func.id}()' is potentially dangerous",
                            'severity': 'high',
                            'category': 'security'
                        })

        return issues

    def analyze_line_metrics(self, code_content):
        """Analyze line-level metrics"""
        lines = code_content.split('\n')

        metrics = {
            'total_lines': len(lines),
            'blank_lines': 0,
            'comment_lines': 0,
            'long_lines': [],
            'avg_line_length': 0
        }

        line_lengths = []

        for i, line in enumerate(lines, 1):
            stripped = line.strip()

            if not stripped:
                metrics['blank_lines'] += 1
            elif stripped.startswith('#'):
                metrics['comment_lines'] += 1

            line_length = len(line)
            line_lengths.append(line_length)

            if line_length > Config.MAX_LINE_LENGTH:
                metrics['long_lines'].append({
                    'line': i,
                    'length': line_length,
                    'content': line[:50] + '...' if len(line) > 50 else line
                })

        if line_lengths:
            metrics['avg_line_length'] = sum(line_lengths) / len(line_lengths)

        return metrics


# 6: AI-POWERED ANALYSIS AGENTS

In [24]:
class AIAnalysisAgents:
    """AI-powered analysis agents using Gemini API"""

    def __init__(self, llm, knowledge_base):
        self.llm = llm
        self.knowledge_base = knowledge_base
        self.setup_prompts()

    def setup_prompts(self):
        """Setup prompt templates for different analysis types"""

        self.code_quality_prompt = PromptTemplate(
            input_variables=["code", "filename", "language", "rules"],
            template="""
You are an expert code reviewer. Analyze the following {language} code for quality, best practices, and potential improvements.

File: {filename}

Code Quality Rules to Consider:
{rules}

Code to Review:
```{language}
{code}
```

Please provide a structured analysis covering:
1. **Code Quality Issues**: Identify violations of coding standards and best practices
2. **Potential Bugs**: Point out logical errors, edge cases, or problematic patterns
3. **Security Concerns**: Highlight any security vulnerabilities or risks
4. **Performance Issues**: Identify inefficient code patterns
5. **Maintainability**: Comment on code readability and maintainability

Format your response as structured feedback with specific line references where possible.
Focus on actionable recommendations rather than generic advice.
"""
        )

        self.security_analysis_prompt = PromptTemplate(
            input_variables=["code", "filename", "language"],
            template="""
You are a security expert reviewing code for vulnerabilities and security issues.

File: {filename}
Language: {language}

Code to Review:
```{language}
{code}
```

Analyze this code specifically for:
1. **Input Validation**: Check for proper input sanitization and validation
2. **Injection Attacks**: SQL injection, command injection, code injection risks
3. **Authentication & Authorization**: Security of access controls
4. **Data Exposure**: Hardcoded secrets, sensitive information leakage
5. **Cryptography**: Proper use of encryption and hashing
6. **Error Handling**: Information disclosure through error messages

Provide specific security recommendations with severity levels (Critical/High/Medium/Low).
Include remediation suggestions for each issue found.
"""
        )

        self.performance_analysis_prompt = PromptTemplate(
            input_variables=["code", "filename", "language", "metrics"],
            template="""
You are a performance optimization expert analyzing code efficiency.

File: {filename}
Language: {language}

Code Metrics:
{metrics}

Code to Review:
```{language}
{code}
```

Analyze this code for:
1. **Algorithmic Complexity**: Time and space complexity issues
2. **Data Structure Usage**: Optimal choice of data structures
3. **Loop Optimization**: Inefficient loops and iterations
4. **Memory Usage**: Memory leaks, unnecessary allocations
5. **I/O Operations**: File handling, database queries, network calls
6. **Caching Opportunities**: Where caching could improve performance

Provide specific performance improvement recommendations with estimated impact.
Focus on changes that would provide the most significant performance gains.
"""
        )

    def analyze_code_quality(self, code_content, filename, language):
        """Perform comprehensive code quality analysis with retry"""
        max_retries = 2
        
        for attempt in range(max_retries):
            try:
                relevant_rules = self.get_relevant_rules(['clarity', 'correctness', 'pythonic'])
                rules_text = self.format_rules_for_prompt(relevant_rules)
                
                # Truncate code if too long (>2000 lines)
                code_lines = code_content.split('\n')
                if len(code_lines) > 2000:
                    code_content = '\n'.join(code_lines[:2000]) + '\n... [truncated]'
                
                chain = self.code_quality_prompt | self.llm
                response = chain.invoke({
                    "code": code_content,
                    "filename": filename,
                    "language": language,
                    "rules": rules_text
                })

                response_content = response.content if hasattr(response, 'content') else str(response)
                
                # Check if response is meaningful
                if response_content and len(response_content.strip()) > 50:
                    return self.parse_ai_response(response, 'quality')
                elif attempt < max_retries - 1:
                    print(f"      Retry {attempt + 1}: Response too short, retrying...")
                    continue
                
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"      Retry {attempt + 1}: {str(e)[:100]}")
                    continue
                return {
                    'error': f"AI analysis failed after {max_retries} attempts: {str(e)}",
                    'recommendations': [],
                    'issues': []
                }
    
        return {
            'error': "Failed to get meaningful response",
            'recommendations': [],
            'issues': []
        }

    def analyze_security(self, code_content, filename, language):
        """Perform security-focused analysis"""
        try:
            chain = self.security_analysis_prompt | self.llm
            response = chain.invoke({
                "code": code_content,
                "filename": filename,
                "language": language
            })

            return self.parse_ai_response(response, 'security')

        except Exception as e:
            return {
                'error': f"Security analysis failed: {str(e)}",
                'vulnerabilities': [],
                'recommendations': []
            }

    def analyze_performance(self, code_content, filename, language, metrics):
        """Perform performance-focused analysis"""
        try:
            metrics_text = json.dumps(metrics, indent=2)

            chain = self.performance_analysis_prompt | self.llm
            response = chain.invoke({
                "code": code_content,
                "filename": filename,
                "language": language,
                "metrics": metrics_text
            })

            return self.parse_ai_response(response, 'performance')

        except Exception as e:
            return {
                'error': f"Performance analysis failed: {str(e)}",
                'optimizations': [],
                'recommendations': []
            }

    def get_relevant_rules(self, categories):
        """Get rules relevant to specific categories"""
        relevant_rules = []

        for category in categories:
            rules = self.knowledge_base.get_rules_by_category(category)
            relevant_rules.extend(rules)

        return relevant_rules

    def format_rules_for_prompt(self, rules):
        """Format rules for inclusion in prompts"""
        formatted_rules = []

        for rule in rules:
            formatted_rules.append(f"- **{rule['name']}**: {rule['description']}")

        return '\n'.join(formatted_rules)

    def parse_ai_response(self, response, analysis_type):
        """Parse AI response into structured format with better extraction"""
        
        # Extract string content from AIMessage object
        response_content = response.content if hasattr(response, 'content') else str(response)
        
        parsed = {
            'analysis_type': analysis_type,
            'summary': {
                'key_points': [],
                'issue_count': 0,
                'severity': 'info'
            },
            'detailed_report_path': None,
            'raw_response': response_content  # Keep for now, will be saved separately
        }
    
        if not response or len(response_content.strip()) < 10:
            return parsed
    
        # Extract key points for summary (first 3-5 major findings)
        lines = response_content.split('\n')
        key_points = []
        issue_count = 0
        
        for line in lines:
            line_stripped = line.strip()
            
            # Look for bullet points or numbered items that represent key findings
            if line_stripped.startswith(('- ', '* ', '• ', '1.', '2.', '3.', '4.', '5.')):
                clean_line = re.sub(r'^[\-\*\•\d\.]+\s*', '', line_stripped)
                if len(clean_line) > 20 and len(key_points) < 5:  # Top 5 points only
                    key_points.append(clean_line[:150])  # Truncate to 150 chars
            
            # Count issues
            if any(word in line.lower() for word in ['issue', 'error', 'problem', 'vulnerability', 'warning']):
                issue_count += 1
        
        # Determine severity based on keywords
        content_lower = response_content.lower()
        if any(word in content_lower for word in ['critical', 'severe', 'dangerous', 'vulnerability']):
            severity = 'critical'
        elif any(word in content_lower for word in ['high', 'important', 'significant']):
            severity = 'high'
        elif any(word in content_lower for word in ['medium', 'moderate', 'warning']):
            severity = 'medium'
        else:
            severity = 'low'
        
        parsed['summary'] = {
            'key_points': key_points if key_points else ['Analysis completed with no major findings'],
            'issue_count': issue_count,
            'severity': severity
        }
        
        return parsed



In [25]:
# Medium-Detail Summary Response Generator

class SummaryResponseGenerator:
    """Generates medium-detail summaries for quick insights"""
    
    def __init__(self, llm):
        self.llm = llm
        self.summary_prompt = PromptTemplate(
            input_variables=["filename", "language", "analysis_data"],
            template="""
You are a senior code reviewer providing a concise but informative summary.

File: {filename} ({language})
Analysis Data: {analysis_data}

Provide a medium-detail summary (300-500 words) covering:

##  Key Issues Found
- List 3-5 most critical issues with severity levels

##  Recommendations  
- Priority fixes with clear action items
- Focus on high-impact improvements

##  Quick Fixes
- Simple changes that can be implemented immediately
- Low-effort, high-value improvements

##  Code Health Score
- Overall assessment (Excellent/Good/Needs Work/Critical)
- Brief explanation of the score

Keep the tone professional but accessible. Be specific about line numbers when possible.
Format using markdown for better readability.
"""
        )
    
    def generate_summary(self, filename, language, analysis_data):
        """Generate a medium-detail summary for a file"""
        try:
            # Prepare analysis data for the prompt
            summary_data = {
                'quality': analysis_data.get('quality_analysis', {}).get('summary', {}),
                'security': analysis_data.get('security_analysis', {}).get('summary', {}),
                'performance': analysis_data.get('performance_analysis', {}).get('summary', {}),
                'metrics': analysis_data.get('metrics', {}),
                'ast_issues': len(analysis_data.get('ast_analysis', {}).get('issues', []))
            }
            
            chain = self.summary_prompt | self.llm
            response = chain.invoke({
                "filename": filename,
                "language": language,
                "analysis_data": str(summary_data)
            })
            
            return response.content if hasattr(response, 'content') else str(response)
            
        except Exception as e:
            return f"Error generating summary for {filename}: {str(e)}"


# 7: REPORT GENERATOR

In [27]:
class ReportGenerator:
    """Generates comprehensive code review reports with multiple output formats"""

    def __init__(self):
        self.report_data = {}
        self.summary_stats = {}

    def generate_comprehensive_report(self, analysis_results):
        """Generate a comprehensive report from all analysis results"""

        report = {
            'metadata': self.generate_metadata(analysis_results),
            'executive_summary': self.generate_executive_summary(analysis_results),
            'detailed_analysis': self.generate_detailed_analysis(analysis_results),
            'recommendations': self.generate_recommendations(analysis_results),
            'metrics_summary': self.generate_metrics_summary(analysis_results)
        }

        return report

    def generate_metadata(self, analysis_results):
        """Generate report metadata"""
        total_files = len(analysis_results)
        total_lines = sum(result.get('metrics', {}).get('total_lines', 0) for result in analysis_results.values())
        languages = set(result.get('language', 'unknown') for result in analysis_results.values())

        return {
            'generated_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_files_analyzed': total_files,
            'total_lines_of_code': total_lines,
            'languages_detected': list(languages),
            'analysis_version': '1.0.0'
        }

    def generate_executive_summary(self, analysis_results):
        """Generate executive summary with key findings"""

        total_issues = 0
        severity_counts = {'high': 0, 'medium': 0, 'low': 0}
        category_counts = defaultdict(int)

        for filename, result in analysis_results.items():
            # Count AST issues
            if 'ast_analysis' in result:
                issues = result['ast_analysis'].get('issues', [])
                total_issues += len(issues)

                for issue in issues:
                    severity = issue.get('severity', 'medium')
                    category = issue.get('category', 'general')

                    if severity in severity_counts:
                        severity_counts[severity] += 1
                    category_counts[category] += 1

            # Count AI-identified issues
            for analysis_type in ['quality_analysis', 'security_analysis', 'performance_analysis']:
                if analysis_type in result:
                    ai_issues = result[analysis_type].get('issues', [])
                    total_issues += len(ai_issues)

        summary = {
            'total_issues_found': total_issues,
            'severity_breakdown': severity_counts,
            'category_breakdown': dict(category_counts),
            'overall_score': self.calculate_overall_score(severity_counts, total_issues),
            'key_findings': self.extract_key_findings(analysis_results)
        }

        return summary

    def calculate_overall_score(self, severity_counts, total_issues):
        """Calculate an overall code quality score (0-100)"""
        if total_issues == 0:
            return 100

        # Weight issues by severity
        weighted_issues = (
            severity_counts['high'] * 3 +
            severity_counts['medium'] * 2 +
            severity_counts['low'] * 1
        )

        # Normalize to 0-100 scale (this is a simple heuristic)
        max_possible_score = 100
        penalty = min(weighted_issues * 5, max_possible_score)

        return max(0, max_possible_score - penalty)

    def extract_key_findings(self, analysis_results):
        """Extract the most important findings across all files"""
        findings = []

        # Look for high-severity security issues
        for filename, result in analysis_results.items():
            if 'security_analysis' in result:
                security_issues = result['security_analysis'].get('issues', [])
                critical_security = [issue for issue in security_issues if 'critical' in issue.lower() or 'high' in issue.lower()]
                if critical_security:
                    findings.append(f"Critical security issues found in {filename}")

            # Look for performance bottlenecks
            if 'performance_analysis' in result:
                perf_issues = result['performance_analysis'].get('issues', [])
                if perf_issues:
                    findings.append(f"Performance optimizations available in {filename}")

            # Look for structural issues
            if 'ast_analysis' in result:
                complexity = result['ast_analysis'].get('complexity', 0)
                if complexity > Config.MAX_COMPLEXITY:
                    findings.append(f"High complexity detected in {filename} (complexity: {complexity})")

        return findings[:5]  # Return top 5 findings

    def generate_detailed_analysis(self, analysis_results):
        """Generate detailed analysis for each file"""
        detailed = {}

        for filename, result in analysis_results.items():
            file_analysis = {
                'filename': filename,
                'language': result.get('language', 'unknown'),
                'file_size': result.get('size', 0),
                'metrics': result.get('metrics', {}),
                'issues_found': [],
                'ai_insights': {},
                'recommendations': []
            }

            # Collect AST analysis issues
            if 'ast_analysis' in result:
                ast_issues = result['ast_analysis'].get('issues', [])
                file_analysis['issues_found'].extend(ast_issues)

            # Collect AI analysis insights
            for analysis_type in ['quality_analysis', 'security_analysis', 'performance_analysis']:
                if analysis_type in result:
                    file_analysis['ai_insights'][analysis_type] = result[analysis_type]

            detailed[filename] = file_analysis

        return detailed

    def generate_recommendations(self, analysis_results):
        """Generate prioritized recommendations"""
        recommendations = {
            'high_priority': [],
            'medium_priority': [],
            'low_priority': [],
            'quick_wins': []
        }

        for filename, result in analysis_results.items():
            # Extract recommendations from AI analysis
            for analysis_type in ['quality_analysis', 'security_analysis', 'performance_analysis']:
                if analysis_type in result:
                    ai_recs = result[analysis_type].get('recommendations', [])

                    for rec in ai_recs:
                        priority = self.determine_recommendation_priority(rec, analysis_type)
                        rec_with_context = f"{filename}: {rec}"

                        if priority == 'high':
                            recommendations['high_priority'].append(rec_with_context)
                        elif priority == 'medium':
                            recommendations['medium_priority'].append(rec_with_context)
                        else:
                            recommendations['low_priority'].append(rec_with_context)

                        # Check if it's a quick win
                        if self.is_quick_win(rec):
                            recommendations['quick_wins'].append(rec_with_context)

        return recommendations

    def determine_recommendation_priority(self, recommendation, analysis_type):
        """Determine the priority of a recommendation"""
        rec_lower = recommendation.lower()

        if analysis_type == 'security_analysis':
            if any(word in rec_lower for word in ['critical', 'vulnerability', 'injection', 'authentication']):
                return 'high'
        elif analysis_type == 'performance_analysis':
            if any(word in rec_lower for word in ['bottleneck', 'optimization', 'slow', 'inefficient']):
                return 'medium'
        elif analysis_type == 'quality_analysis':
            if any(word in rec_lower for word in ['bug', 'error', 'exception', 'crash']):
                return 'high'

        return 'medium'

    def is_quick_win(self, recommendation):
        """Determine if a recommendation is a quick win (easy to implement)"""
        quick_win_keywords = [
            'add docstring', 'rename variable', 'add comment', 'import order',
            'line length', 'whitespace', 'formatting', 'pep 8'
        ]

        rec_lower = recommendation.lower()
        return any(keyword in rec_lower for keyword in quick_win_keywords)

    def generate_metrics_summary(self, analysis_results):
        """Generate summary of code metrics"""
        metrics = {
            'total_files': len(analysis_results),
            'total_lines': 0,
            'total_functions': 0,
            'total_classes': 0,
            'average_complexity': 0,
            'language_distribution': defaultdict(int)
        }

        complexities = []

        for filename, result in analysis_results.items():
            # File metrics
            file_metrics = result.get('metrics', {})
            metrics['total_lines'] += file_metrics.get('total_lines', 0)

            # Language distribution
            language = result.get('language', 'unknown')
            metrics['language_distribution'][language] += 1

            # AST metrics
            if 'ast_analysis' in result:
                ast_data = result['ast_analysis']

                # Functions and classes
                functions = ast_data.get('functions', [])
                classes = ast_data.get('classes', [])

                metrics['total_functions'] += len(functions)
                metrics['total_classes'] += len(classes)

                # Complexity
                complexity = ast_data.get('complexity', 1)
                complexities.append(complexity)

        # Calculate average complexity
        if complexities:
            metrics['average_complexity'] = sum(complexities) / len(complexities)

        # Convert defaultdict to regular dict
        metrics['language_distribution'] = dict(metrics['language_distribution'])

        return metrics

    def display_html_report(self, report):
        """Display formatted HTML report in Colab"""

        html_content = f"""
        <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; max-width: 1200px; margin: 0 auto;">
            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px 10px 0 0;">
                <h1 style="margin: 0; font-size: 2.5em; font-weight: bold;">Code Review Report</h1>
                <p style="margin: 10px 0 0 0; font-size: 1.2em; opacity: 0.9;">Generated on {report['metadata']['generated_at']}</p>
            </div>

            <div style="background: #f8f9fa; padding: 20px; border-left: 5px solid #28a745;">
                <h2 style="color: #28a745; margin-top: 0;">Executive Summary</h2>
                <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-bottom: 20px;">
                    <div style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                        <h4 style="margin: 0; color: #6c757d;">Overall Score</h4>
                        <p style="font-size: 2em; font-weight: bold; margin: 5px 0; color: {'#28a745' if report['executive_summary']['overall_score'] >= 80 else '#ffc107' if report['executive_summary']['overall_score'] >= 60 else '#dc3545'};">
                            {report['executive_summary']['overall_score']}/100
                        </p>
                    </div>
                    <div style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                        <h4 style="margin: 0; color: #6c757d;">Total Issues</h4>
                        <p style="font-size: 2em; font-weight: bold; margin: 5px 0; color: #dc3545;">
                            {report['executive_summary']['total_issues_found']}
                        </p>
                    </div>
                    <div style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                        <h4 style="margin: 0; color: #6c757d;">Files Analyzed</h4>
                        <p style="font-size: 2em; font-weight: bold; margin: 5px 0; color: #17a2b8;">
                            {report['metadata']['total_files_analyzed']}
                        </p>
                    </div>
                </div>
            </div>

            <div style="background: white; padding: 20px; border-left: 5px solid #dc3545;">
                <h2 style="color: #dc3545; margin-top: 0;">Key Findings</h2>
                <ul style="list-style-type: none; padding: 0;">
        """

        for finding in report['executive_summary']['key_findings']:
            html_content += f'<li style="padding: 8px 0; border-bottom: 1px solid #eee;">• {finding}</li>'

        html_content += """
                </ul>
            </div>

            <div style="background: #e8f5e8; padding: 20px; border-left: 5px solid #28a745; margin-top: 20px;">
                <h2 style="color: #28a745; margin-top: 0;">Quick Wins</h2>
                <div style="background: white; padding: 15px; border-radius: 8px;">
                    <ul>
        """

        for quick_win in report['recommendations']['quick_wins'][:5]:
            html_content += f'<li style="margin-bottom: 8px;">{quick_win}</li>'

        html_content += """
                    </ul>
                </div>
            </div>
        </div>
        """

        display(HTML(html_content))

# 8: MAIN APPLICATION ORCHESTRATOR

In [29]:
class CodeReviewBuddy:
    """Main application class that orchestrates the entire code review process"""

    def __init__(self):
        self.file_handler = LocalFileHandler()
        self.analyzer = CodeAnalyzer()
        self.ai_agents = AIAnalysisAgents(llm, knowledge_base)
        self.report_generator = ReportGenerator()
        self.summary_generator = SummaryResponseGenerator(llm)
        
        print("Code Review Buddy initialized successfully!")
        print("Knowledge base loaded with comprehensive rules")
        print("AI agents ready for deep analysis")

    def welcome_message(self):
        """Display welcome message and instructions"""
        welcome_html = """
        <div style="font-family: Arial, sans-serif; max-width: 800px; margin: 20px auto; padding: 20px; background: linear-gradient(135deg, #74b9ff, #0984e3); color: white; border-radius: 15px; box-shadow: 0 8px 25px rgba(0,0,0,0.15);">
            <div style="text-align: center;">
                <h1 style="font-size: 3em; margin-bottom: 10px;">Code Review Buddy</h1>
                <p style="font-size: 1.3em; margin-bottom: 20px; opacity: 0.9;">Your AI-Powered Code Analysis Companion</p>
            </div>
    
            <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 10px; margin: 20px 0;">
                <h3 style="margin-top: 0;">What I Can Do For You:</h3>
                <ul style="font-size: 1.1em; line-height: 1.6;">
                    <li><strong>Deep Code Analysis</strong> - AST parsing and structural analysis</li>
                    <li><strong>Security Review</strong> - Identify vulnerabilities and risks</li>
                    <li><strong>Performance Optimization</strong> - Find bottlenecks and inefficiencies</li>
                    <li><strong>Quality Assessment</strong> - PEP 8, best practices, maintainability</li>
                    <li><strong>AI-Powered Insights</strong> - Gemini AI analysis and recommendations</li>
                    <li><strong>Comprehensive Reports</strong> - Beautiful, actionable reports</li>
                    <li><strong>Medium-Detail Summaries</strong> - Quick insights without information overload</li>
                </ul>
            </div>
    
            <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 10px;">
                <h3 style="margin-top: 0;">Supported Languages & Features:</h3>
                <p><strong>Languages:</strong> Python, JavaScript, Java, C++, TypeScript, PHP, Ruby, Go, Rust</p>
                <p><strong>Upload Methods:</strong> Individual files, ZIP archives, GitHub repositories</p>
                <p><strong>Analysis Types:</strong> Static analysis, AI review, security audit, performance profiling</p>
            </div>
        </div>
        """
    
        display(HTML(welcome_html))
        
        # Add enhanced options display
        self.display_enhanced_options()

    # ADD THIS NEW METHOD TO CodeReviewBuddy CLASS IN CELL 22
    def display_enhanced_options(self):
        """Display enhanced user options with all available features"""
    
        options_html = """
        <div style="font-family: 'Segoe UI', sans-serif; max-width: 900px; margin: 20px auto; 
                    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    border-radius: 15px; padding: 25px; color: white;">
            
            <h2 style="text-align: center; margin-bottom: 30px; font-size: 2.2em;">
                Ready to Analyze Your Code!
            </h2>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 25px;">
                
                <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 10px;">
                    <h3 style="margin-top: 0; color: #FFE066;">Quick Start</h3>
                    <div style="font-family: 'Courier New', monospace; background: rgba(0,0,0,0.3); 
                               padding: 10px; border-radius: 5px; margin: 10px 0;">
                        <code style="color: #4CAF50;">report, results = main()</code>
                    </div>
                    <p style="font-size: 0.9em; margin-bottom: 0;">
                        Complete analysis with interactive file selection
                    </p>
                </div>
                
                <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 10px;">
                    <h3 style="margin-top: 0; color: #FFE066;">Advanced Control</h3>
                    <div style="font-family: 'Courier New', monospace; background: rgba(0,0,0,0.3); 
                               padding: 10px; border-radius: 5px; margin: 10px 0;">
                        <code style="color: #4CAF50;">app = CodeReviewBuddy()<br>
                        report, results = app.run_analysis()</code>
                    </div>
                    <p style="font-size: 0.9em; margin-bottom: 0;">
                        Step-by-step control over the analysis process
                    </p>
                </div>
            </div>
            
            <div style="background: rgba(255,255,255,0.15); padding: 20px; border-radius: 10px; margin-bottom: 20px;">
                <h3 style="margin-top: 0; color: #FFE066;">New Analysis Features</h3>
                
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px;">
                    <div>
                        <h4 style="color: #B8E6B8; margin: 10px 0 5px 0;">Get Quick Summaries</h4>
                        <div style="font-family: 'Courier New', monospace; background: rgba(0,0,0,0.3); 
                                   padding: 8px; border-radius: 5px; font-size: 0.85em;">
                            <code style="color: #4CAF50;">summaries = get_file_summaries(results, app)</code>
                        </div>
                    </div>
                    
                    <div>
                        <h4 style="color: #B8E6B8; margin: 10px 0 5px 0;">View Detailed Reports</h4>
                        <div style="font-family: 'Courier New', monospace; background: rgba(0,0,0,0.3); 
                                   padding: 8px; border-radius: 5px; font-size: 0.85em;">
                            <code style="color: #4CAF50;">view_detailed_reports()</code>
                        </div>
                    </div>
                    
                    <div>
                        <h4 style="color: #B8E6B8; margin: 10px 0 5px 0;">Focus on Security</h4>
                        <div style="font-family: 'Courier New', monospace; background: rgba(0,0,0,0.3); 
                                   padding: 8px; border-radius: 5px; font-size: 0.85em;">
                            <code style="color: #4CAF50;">security_report = get_security_focus(results)</code>
                        </div>
                    </div>
                    
                    <div>
                        <h4 style="color: #B8E6B8; margin: 10px 0 5px 0;">Get Quick Fixes</h4>
                        <div style="font-family: 'Courier New', monospace; background: rgba(0,0,0,0.3); 
                                   padding: 8px; border-radius: 5px; font-size: 0.85em;">
                            <code style="color: #4CAF50;">fixes = get_quick_fixes(results)</code>
                        </div>
                    </div>
                </div>
            </div>
            
            <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 10px;">
                <h3 style="margin-top: 0; color: #FFE066;">Supported Upload Methods</h3>
                <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; text-align: center;">
                    <div style="padding: 10px;">
                        <div style="font-size: 2em; margin-bottom: 5px;"></div>
                        <div style="font-size: 0.9em;">Individual Files</div>
                    </div>
                    <div style="padding: 10px;">
                        <div style="font-size: 2em; margin-bottom: 5px;"></div>
                        <div style="font-size: 0.9em;">ZIP Archives</div>
                    </div>
                    <div style="padding: 10px;">
                        <div style="font-size: 2em; margin-bottom: 5px;"></div>
                        <div style="font-size: 0.9em;">GitHub Repos</div>
                    </div>
                    <div style="padding: 10px;">
                        <div style="font-size: 2em; margin-bottom: 5px;"></div>
                        <div style="font-size: 0.9em;">Directories</div>
                    </div>
                </div>
            </div>
            
            <div style="text-align: center; margin-top: 20px; font-size: 0.9em; opacity: 0.9;">
                <strong>Supported Languages:</strong> Python • JavaScript • Java • C++ • TypeScript • PHP • Ruby • Go • Rust
            </div>
        </div>
        """
    
        display(HTML(options_html))
    
    def run_analysis(self):
        """Main method to run the complete code analysis process"""

        # Display welcome message
        self.welcome_message()

        try:
            # Step 1: File Upload and Processing
            print("\n" + "="*80)
            print("STEP 1: FILE UPLOAD AND PROCESSING")
            print("="*80)

            processed_files = self.file_handler.process_files()

            if not processed_files:
                print("No files to analyze. Exiting...")
                return

            print(f"\nSuccessfully processed {len(processed_files)} files")

            # Step 2: Code Analysis
            print("\n" + "="*80)
            print("STEP 2: COMPREHENSIVE CODE ANALYSIS")
            print("="*80)

            analysis_results = {}

            for i, (filename, file_data) in enumerate(processed_files.items(), 1):
                print(f"\nAnalyzing file {i}/{len(processed_files)}: {filename}")

                file_analysis = self.analyze_single_file(filename, file_data)
                analysis_results[filename] = file_analysis

                # Show progress
                progress = (i / len(processed_files)) * 100
                print(f"Progress: {progress:.1f}% complete")

            # Step 3: Generate Reports
            print("\n" + "="*80)
            print("STEP 3: GENERATING COMPREHENSIVE REPORT")
            print("="*80)

            final_report = self.report_generator.generate_comprehensive_report(analysis_results)

            # Step 4: Display Results
            print("\n" + "="*80)
            print("STEP 4: ANALYSIS COMPLETE!")
            print("="*80)

            self.display_results(final_report, analysis_results)

            return final_report, analysis_results

        except Exception as e:
            print(f"Analysis failed: {str(e)}")
            import traceback
            traceback.print_exc()
            return None, None

    def analyze_single_file(self, filename, file_data):
        """Analyze a single file comprehensively"""

        # Initialize report manager
        report_manager = DetailedReportManager()
    
        analysis_result = {
            'filename': filename,
            'language': file_data['language'],
            'size': file_data['size'],
            'path': file_data['path']
        }

        code_content = file_data['content']
        language = file_data['language']

        try:
            # 1. Line-level metrics
            print(f"   Analyzing line metrics...")
            analysis_result['metrics'] = self.analyzer.analyze_line_metrics(code_content)

            # 2. AST Analysis (for Python)
            if language == 'python':
                print(f"   Performing AST analysis...")
                analysis_result['ast_analysis'] = self.analyzer.analyze_python_ast(code_content, filename)

            # 3. AI-Powered Quality Analysis
            print(f"   Running AI quality analysis...")
            quality_analysis = self.ai_agents.analyze_code_quality(
                code_content, filename, language
            )
        
            # Save detailed report and keep only summary
            if 'raw_response' in quality_analysis:
                quality_analysis['detailed_report_path'] = report_manager.save_detailed_report(
                    filename, 'quality', quality_analysis['raw_response']
                )
                del quality_analysis['raw_response']  # Remove verbose content from main JSON
        
            analysis_result['quality_analysis'] = quality_analysis
    
            # 4. Security Analysis
            print(f"   Performing security analysis...")
            security_analysis = self.ai_agents.analyze_security(
                code_content, filename, language
            )
        
            if 'raw_response' in security_analysis:
                security_analysis['detailed_report_path'] = report_manager.save_detailed_report(
                    filename, 'security', security_analysis['raw_response']
                )
                del security_analysis['raw_response']
        
            analysis_result['security_analysis'] = security_analysis
    
            # 5. Performance Analysis
            print(f"   Analyzing performance...")
            performance_analysis = self.ai_agents.analyze_performance(
                code_content, filename, language, analysis_result.get('metrics', {})
            )
            
            if 'raw_response' in performance_analysis:
                performance_analysis['detailed_report_path'] = report_manager.save_detailed_report(
                    filename, 'performance', performance_analysis['raw_response']
                )
                del performance_analysis['raw_response']
        
            analysis_result['performance_analysis'] = performance_analysis

            print(f"   Analysis complete for {filename}")

        except Exception as e:
            print(f"   Error analyzing {filename}: {str(e)}")
            analysis_result['error'] = str(e)

        return analysis_result

    def display_results(self, report, analysis_results):
        """Display comprehensive analysis results"""

        # Display HTML report
        self.report_generator.display_html_report(report)
    
    # Display detailed findings for each file
        print("\n" + "="*100)
        print("DETAILED ANALYSIS RESULTS")
        print("="*100)
    
        for filename, result in analysis_results.items():
            self.display_file_analysis(filename, result)
    
        # Display summary statistics
        self.display_summary_statistics(report)
        
        # Save JSON report
        json_path = self.save_json_report(report, analysis_results)
        
        print(f"\nAnalysis complete!")
        print(f"   - Main JSON report: {json_path}")
        print(f"   - Detailed markdown reports: code_review_reports/")
        
    def display_file_analysis(self, filename, result):
        """Display detailed analysis for a single file with improved formatting"""
    
        print(f"\n{'='*80}")
        print(f"FILE: {filename}")
        print(f"{'='*80}")
    
        # Metadata
        language = result.get('language', 'unknown')
        size_kb = result.get('size', 0) / 1024
        print(f"\nMETADATA")
        print(f"  Language: {language}")
        print(f"  Size: {size_kb:.1f} KB")
    
        # Metrics
        if 'metrics' in result:
            metrics = result['metrics']
            print(f"\nMETRICS")
            print(f"  Total Lines: {metrics.get('total_lines', 0)}")
            print(f"  Avg Line Length: {metrics.get('avg_line_length', 0):.1f}")
            print(f"  Long Lines: {len(metrics.get('long_lines', []))}")
            print(f"  Comment Lines: {metrics.get('comment_lines', 0)}")
    
        # AST Analysis
        if 'ast_analysis' in result:
            ast_data = result['ast_analysis']
            if 'error' not in ast_data:
                print(f"\nSTATIC ANALYSIS")
                print(f"  Complexity: {ast_data.get('complexity', 0)}")
                print(f"  Functions: {len(ast_data.get('functions', []))}")
                print(f"  Classes: {len(ast_data.get('classes', []))}")
                print(f"  Issues Found: {len(ast_data.get('issues', []))}")
    
        # AI Analysis Summaries
        ai_sections = {
            'quality_analysis': ' CODE QUALITY',
            'security_analysis': ' SECURITY',
            'performance_analysis': ' PERFORMANCE'
        }
    
        for section_key, section_title in ai_sections.items():
            if section_key in result and 'error' not in result[section_key]:
                analysis = result[section_key]
                summary = analysis.get('summary', {})
                
                print(f"\n{section_title}")
                print(f"  Severity: {summary.get('severity', 'info').upper()}")
                print(f"  Issues Found: {summary.get('issue_count', 0)}")
                
                print(f"  Key Findings:")
                for i, point in enumerate(summary.get('key_points', [])[:3], 1):
                    print(f"    {i}. {point}")
                
                # Show path to detailed report
                if 'detailed_report_path' in analysis:
                    print(f"   Detailed Report: {analysis['detailed_report_path']}")
    
        print(f"\n{'='*80}\n")

    def save_json_report(self, report, analysis_results, filename="analysis_summary.json"):
        """Save the main analysis summary as JSON"""
        output_dir = Path("code_review_reports")
        output_dir.mkdir(exist_ok=True)
        
        timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
        json_path = output_dir / f"{filename.replace('.json', '')}_{timestamp}.json"
        
        # Combine report and results
        full_report = {
            'metadata': report['metadata'],
            'executive_summary': report['executive_summary'],
            'metrics_summary': report['metrics_summary'],
            'files': analysis_results
        }
    
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(full_report, f, indent=2, ensure_ascii=False)
        
        print(f"\n Main report saved: {json_path}")
        return str(json_path)
    
    def display_summary_statistics(self, report):
        """Display overall summary statistics"""

        print("\n" + "="*100)
        print("SUMMARY STATISTICS")
        print("="*100)

        metadata = report['metadata']
        summary = report['executive_summary']
        metrics = report['metrics_summary']

        # Create summary table
        summary_data = [
            ['Total Files Analyzed', metadata['total_files_analyzed']],
            ['Total Lines of Code', f"{metadata['total_lines_of_code']:,}"],
            ['Languages Detected', ', '.join(metadata['languages_detected'])],
            ['Overall Quality Score', f"{summary['overall_score']}/100"],
            ['Total Issues Found', summary['total_issues_found']],
            ['High Severity Issues', summary['severity_breakdown']['high']],
            ['Medium Severity Issues', summary['severity_breakdown']['medium']],
            ['Low Severity Issues', summary['severity_breakdown']['low']],
            ['Average Complexity', f"{metrics['average_complexity']:.1f}"],
            ['Total Functions', metrics['total_functions']],
            ['Total Classes', metrics['total_classes']]
        ]

        print(tabulate(summary_data, headers=['Metric', 'Value'], tablefmt='grid'))

        # Language distribution
        if metrics['language_distribution']:
            print(f"\nLanguage Distribution:")
            lang_data = [[lang, count] for lang, count in metrics['language_distribution'].items()]
            print(tabulate(lang_data, headers=['Language', 'Files'], tablefmt='grid'))


# 9: APPLICATION LAUNCHER

In [31]:
def main():
    """Main function to launch Code Review Buddy"""

    # Initialize the application
    app = CodeReviewBuddy()

    # Run the analysis
    report, results = app.run_analysis()

    if report and results:
        print("\nAnalysis completed successfully!")
        print("Results are available in the 'report' and 'results' variables")
        print("You can access individual file analysis using: results['filename']")
        print("Full report data is available in: report")

        return report, results
    else:
        print("Analysis was not completed successfully")
        return None, None


## 9.5: HELPER FUNCTIONS FOR ENHANCED USER EXPERIENCE

In [33]:
# Helper function for summaries

def get_file_summaries(results, app):
    """Generate and display medium-detail summaries for all files"""
    
    print("\n" + "="*100)
    print("FILE SUMMARIES - MEDIUM DETAIL ANALYSIS")
    print("="*100)
    
    summaries = {}
    
    for filename, file_data in results.items():
        print(f"\n{'='*80}")
        print(f"SUMMARY: {filename}")
        print(f"{'='*80}")
        
        language = file_data.get('language', 'unknown')
        summary = app.summary_generator.generate_summary(filename, language, file_data)
        summaries[filename] = summary
        
        display(Markdown(summary))
        print("\n" + "-"*80)
    
    return summaries

In [34]:
# Helper function for security focus

def get_security_focus(results):
    """Extract and display security-focused analysis"""
    
    print("\n" + "="*100)
    print("SECURITY FOCUS REPORT")
    print("="*100)
    
    security_issues = {}
    
    for filename, file_data in results.items():
        if 'security_analysis' in file_data:
            security_data = file_data['security_analysis']
            summary = security_data.get('summary', {})
            
            print(f"\n{filename}")
            print(f"   Severity: {summary.get('severity', 'unknown').upper()}")
            print(f"   Issues Found: {summary.get('issue_count', 0)}")
            
            for i, point in enumerate(summary.get('key_points', [])[:3], 1):
                print(f"   {i}. {point[:100]}...")
            
            if 'detailed_report_path' in security_data:
                print(f"   Detailed Report: {security_data['detailed_report_path']}")
            
            security_issues[filename] = security_data
    
    return security_issues

In [35]:
# Helper function for quick fixes

def get_quick_fixes(results):
    """Extract quick fixes and actionable recommendations"""
    
    print("\n" + "="*100)
    print("QUICK FIXES & IMMEDIATE ACTIONS")
    print("="*100)
    
    quick_fixes = {}
    
    for filename, file_data in results.items():
        file_fixes = []
        
        if 'quality_analysis' in file_data:
            quality_summary = file_data['quality_analysis'].get('summary', {})
            for point in quality_summary.get('key_points', []):
                if any(keyword in point.lower() for keyword in 
                      ['fix', 'change', 'replace', 'add', 'remove', 'update']):
                    file_fixes.append(f"Quality: {point[:150]}")
        
        if 'ast_analysis' in file_data:
            ast_issues = file_data['ast_analysis'].get('issues', [])
            for issue in ast_issues:
                if issue.get('severity') in ['high', 'medium']:
                    file_fixes.append(f"Static: {issue.get('message', '')}")
        
        if file_fixes:
            print(f"\n{filename}")
            for i, fix in enumerate(file_fixes[:5], 1):
                print(f"   {i}. {fix}")
            quick_fixes[filename] = file_fixes
    
    return quick_fixes

In [36]:
#Helper function for viewing reports

def view_detailed_reports():
    """List and provide access to detailed markdown reports"""
    
    print("\n" + "="*100)
    print("DETAILED REPORTS AVAILABLE")
    print("="*100)
    
    reports_dir = Path("code_review_reports")
    if reports_dir.exists():
        markdown_files = list(reports_dir.glob("*.md"))
        json_files = list(reports_dir.glob("*.json"))
        
        if markdown_files or json_files:
            print(f"\nReports Directory: {reports_dir}")
            
            if markdown_files:
                print(f"\nDetailed Analysis Reports ({len(markdown_files)} files):")
                for report_file in sorted(markdown_files):
                    print(f"   - {report_file.name}")
            
            if json_files:
                print(f"\nJSON Summary Reports ({len(json_files)} files):")
                for json_file in sorted(json_files):
                    print(f"   - {json_file.name}")
            
            print(f"\nAccess reports using:")
            print(f"   with open('code_review_reports/filename.md', 'r') as f:")
            print(f"       content = f.read()")
            print(f"       display(Markdown(content))")
        else:
            print("No reports found. Run analysis first.")
    else:
        print("Reports directory not found. Run analysis first.")

# 10: INTERACTIVE EXECUTION

In [38]:
# startup display

print("="*80)
print("CODE REVIEW BUDDY - READY TO ANALYZE!")
print("="*80)

print("\nQUICK START OPTIONS:")
print("  1. Complete Analysis:  report, results = main()")
print("  2. Step-by-step:       app = CodeReviewBuddy()")
print("                         report, results = app.run_analysis()")

print("\nANALYSIS TOOLS (Use after running main()):")
print("  - Medium Summaries:    summaries = get_file_summaries(results, app)")
print("  - Security Focus:      security = get_security_focus(results)")
print("  - Quick Fixes:         fixes = get_quick_fixes(results)")
print("  - View Reports:        view_detailed_reports()")
print("  - Individual File:     results['filename.py']")
print("  - Full Report:         report")

print("\n" + "="*80)
print("Tip: After analysis, call get_file_summaries(results, app)")
print("     for medium-detail summaries of all files!")
print("="*80)

CODE REVIEW BUDDY - READY TO ANALYZE!

QUICK START OPTIONS:
  1. Complete Analysis:  report, results = main()
  2. Step-by-step:       app = CodeReviewBuddy()
                         report, results = app.run_analysis()

ANALYSIS TOOLS (Use after running main()):
  - Medium Summaries:    summaries = get_file_summaries(results, app)
  - Security Focus:      security = get_security_focus(results)
  - Quick Fixes:         fixes = get_quick_fixes(results)
  - View Reports:        view_detailed_reports()
  - Individual File:     results['filename.py']
  - Full Report:         report

Tip: After analysis, call get_file_summaries(results, app)
     for medium-detail summaries of all files!


In [39]:
# Run the analysis and save the app instance
app = CodeReviewBuddy()
# report, results = app.run_analysis()

# Now you can use:
# summaries = get_file_summaries(results, app)
# security = get_security_focus(results)
# fixes = get_quick_fixes(results)

Code Review Buddy initialized successfully!
Knowledge base loaded with comprehensive rules
AI agents ready for deep analysis


# Final Test Run

### 1.web_scraper.py

In [43]:
app = CodeReviewBuddy()
report, results = app.run_analysis()

Code Review Buddy initialized successfully!
Knowledge base loaded with comprehensive rules
AI agents ready for deep analysis



STEP 1: FILE UPLOAD AND PROCESSING

CODE UPLOAD OPTIONS
1. Upload Individual Files (specify paths)
2. Upload ZIP Archive
3. Clone GitHub Repository
4. Scan Directory



Enter your choice (1-4):  1



Enter file paths (one per line, empty line to finish):
Example: /path/to/your/file.py


File path:  C:\Users\saumy\Desktop\ML & AI Upgrade IIIT\GenAI\Semantic Spotter - Project\BYOP - Code Review Buddy\Test codes\web_scraper.py
File path:  


 Processed web_scraper.py (3.3KB)

Successfully processed 1 files

STEP 2: COMPREHENSIVE CODE ANALYSIS

Analyzing file 1/1: web_scraper.py
   Analyzing line metrics...
   Performing AST analysis...
   Running AI quality analysis...
   Performing security analysis...
   Analyzing performance...
   Analysis complete for web_scraper.py
Progress: 100.0% complete

STEP 3: GENERATING COMPREHENSIVE REPORT

STEP 4: ANALYSIS COMPLETE!



DETAILED ANALYSIS RESULTS

FILE: web_scraper.py

METADATA
  Language: python
  Size: 3.3 KB

METRICS
  Total Lines: 102
  Avg Line Length: 31.1
  Long Lines: 0
  Comment Lines: 5

STATIC ANALYSIS
  Complexity: 9
  Functions: 7
  Classes: 1
  Issues Found: 0

 CODE QUALITY
  Severity: CRITICAL
  Issues Found: 14
  Key Findings:
    1. **[Minor] Line 84: `import json` inside a method.**
    2. **Issue**: Imports should be placed at the top of the file (per PEP 8). Placing them inside a function can lead to confusion and slight performance ov
    3. **Recommendation**: Move `import json` to the top of the file with the other imports.
   Detailed Report: code_review_reports\web_scraper.py_quality_20250930_001613.md

 SECURITY
  Severity: CRITICAL
  Issues Found: 15
  Key Findings:
    1. **Severity**: **High**
    2. **Location**: `fetch_page(self, url: str)` function, line 23. This is also exposed through `scrape_multiple_pages`.
    3. **Description**: The `fetch_page` function accepts 

In [45]:
summaries = get_file_summaries(results, app)


FILE SUMMARIES - MEDIUM DETAIL ANALYSIS

SUMMARY: web_scraper.py


Here is a concise but informative summary of the code review for `web_scraper.py`.

***

### Code Review Summary: `web_scraper.py`

This review provides an analysis of the `web_scraper.py` file, highlighting critical issues and offering recommendations for improvement. While the script's intent is clear, several significant security and performance vulnerabilities require immediate attention.

### Key Issues Found

1.  **Server-Side Request Forgery (SSRF) (Severity: High)**: The `fetch_page` function (line 23) directly uses a user-provided URL without validation. This allows an attacker to force the server to make requests to internal network resources or arbitrary external sites, posing a major security risk.
2.  **Risk of Catastrophic Backtracking (Severity: Critical)**: The `extract_links` function uses regular expressions to parse HTML. Complex or maliciously crafted HTML can trigger "catastrophic backtracking," causing the regex engine to consume excessive CPU and effectively freeze the application (a ReDoS vulnerability).
3.  **Unbounded Memory Growth (Severity: Critical)**: The scraper appends all results to the `self.results` list. For a large number of URLs, this will lead to high memory consumption, potentially causing the application to crash due to an out-of-memory error.
4.  **Overly Broad Exception Handling (Severity: Minor)**: The `try...except` block on line 90 catches the generic `Exception`. This practice is discouraged as it can hide underlying bugs, suppress important system-level exceptions, and make debugging significantly more difficult.

### Recommendations

**Priority Fixes:**

*   **Mitigate SSRF Vulnerability**: The highest priority is to secure the `fetch_page` function.
    *   **Action**: Implement a strict allowlist of domains that the scraper is authorized to contact.
    *   **Action**: Validate the URL scheme, ensuring it is only `http` or `https` before making the request. Reject all other schemes (e.g., `file://`, `ftp://`).
*   **Use a Robust HTML Parser**: To eliminate the ReDoS risk and improve reliability, replace the regex-based link extraction.
    *   **Action**: Refactor `extract_links` to use a dedicated HTML parsing library like `BeautifulSoup` or `lxml`. These tools are designed to handle complex and malformed HTML safely and efficiently.
*   **Implement Scalable Data Handling**: To prevent memory exhaustion, change how results are stored.
    *   **Action**: Instead of accumulating results in a list, modify the scraper to stream data directly to a persistent storage solution. Writing each result as a line in a JSONL file or inserting it into a database are excellent alternatives.

### Quick Fixes

*   **Standardize Imports**: Move `import json` from line 84 to the top of the file with the other imports. This adheres to PEP 8 standards and improves code readability.
*   **Specify Exceptions**: On line 90, replace `except Exception as e` with more specific exceptions relevant to file operations, such as `except (IOError, OSError) as e`.

### Code Health Score

**Score: Critical**

The codebase contains a high-severity security vulnerability (SSRF) and critical performance flaws that could lead to a denial of service. The high number of quality and security issues relative to the file's small size indicates that fundamental best practices were overlooked. These issues must be addressed before the code can be considered for a production environment.


--------------------------------------------------------------------------------


### 2.data_processor.py

In [48]:
app = CodeReviewBuddy()
report, results = app.run_analysis()

Code Review Buddy initialized successfully!
Knowledge base loaded with comprehensive rules
AI agents ready for deep analysis



STEP 1: FILE UPLOAD AND PROCESSING

CODE UPLOAD OPTIONS
1. Upload Individual Files (specify paths)
2. Upload ZIP Archive
3. Clone GitHub Repository
4. Scan Directory



Enter your choice (1-4):  1



Enter file paths (one per line, empty line to finish):
Example: /path/to/your/file.py


File path:  C:\Users\saumy\Desktop\ML & AI Upgrade IIIT\GenAI\Semantic Spotter - Project\BYOP - Code Review Buddy\Test codes\data_processor.py
File path:  


 Processed data_processor.py (2.9KB)

Successfully processed 1 files

STEP 2: COMPREHENSIVE CODE ANALYSIS

Analyzing file 1/1: data_processor.py
   Analyzing line metrics...
   Performing AST analysis...
   Running AI quality analysis...
   Performing security analysis...
   Analyzing performance...
   Analysis complete for data_processor.py
Progress: 100.0% complete

STEP 3: GENERATING COMPREHENSIVE REPORT

STEP 4: ANALYSIS COMPLETE!



DETAILED ANALYSIS RESULTS

FILE: data_processor.py

METADATA
  Language: python
  Size: 2.9 KB

METRICS
  Total Lines: 93
  Avg Line Length: 30.2
  Long Lines: 0
  Comment Lines: 8

STATIC ANALYSIS
  Complexity: 20
  Functions: 7
  Classes: 1
  Issues Found: 0

 CODE QUALITY
  Severity: CRITICAL
  Issues Found: 22
  Key Findings:
    1. **`__init__` (Line 6): Unused Class Attribute**
    2. **Issue**: The `self.cache` attribute is initialized but never used anywhere in the class. This is dead code and adds unnecessary complexity.
    3. **Recommendation**: Remove the `self.cache = []` line.
   Detailed Report: code_review_reports\data_processor.py_quality_20250930_001937.md

 SECURITY
  Severity: CRITICAL
  Issues Found: 27
  Key Findings:
    1. **Severity:** <span style="color:red">**High**</span>
    2. **Vulnerability Category:** Input Validation
    3. **Description:** The `load_large_dataset` method accepts a `filepath` parameter and uses it directly in the `open()` function wit

In [50]:
summaries = get_file_summaries(results, app)


FILE SUMMARIES - MEDIUM DETAIL ANALYSIS

SUMMARY: data_processor.py


Here is a concise code review summary for `data_processor.py`.

***

### Code Review Summary: `data_processor.py`

This review highlights critical security, performance, and quality issues found in the `data_processor.py` file. The code contains a severe vulnerability and significant performance bottlenecks that must be addressed before it can be considered for production use.

---

### Key Issues Found

-   **Critical: Path Traversal Security Vulnerability:** The `load_large_dataset` method accepts a `filepath` and uses it to open a file directly. This allows an attacker to provide a malicious path (e.g., `../../etc/passwd`) to access sensitive files outside the intended directory.
-   **High: Severe Performance Bottlenecks:** The code contains multiple algorithms with quadratic time complexity (O(n²)). The `find_duplicates` and `calculate_statistics` functions use nested loops and a manual sort, which will be extremely slow and unusable for large datasets.
-   **High: Inefficient Memory Usage:** The `load_large_dataset` function loads the entire JSON file into memory at once. This will lead to high memory consumption and potential application crashes when processing genuinely large files.
-   **Medium: Unused Code and Unpythonic Patterns:** The class contains dead code, such as the unused `self.cache` attribute initialized in `__init__` (Line 6). Additionally, the loop on Line 16 to check for a key's existence is redundant and unpythonic.

### Recommendations

The following fixes are prioritized based on their impact on security and stability.

1.  **Address the Path Traversal Vulnerability Immediately:** This is the highest priority. In `load_large_dataset`, you must validate the user-provided `filepath`.
    -   **Action:** Define a secure base directory for data files. Before opening, resolve the `filepath` to an absolute path and verify it is located within this base directory. Reject any path that attempts to navigate outside of it.

2.  **Refactor Inefficient Algorithms for Performance:** The current O(n²) implementations will not scale.
    -   **Action (`find_duplicates`):** Replace the nested loop with a more efficient O(n) approach. Iterate through the list once, using a `set` or `collections.Counter` to track seen items and identify duplicates.
    -   **Action (`calculate_statistics`):** Replace the manual sorting algorithm with Python's highly optimized built-in `list.sort()` or `sorted()` function, which uses Timsort (O(n log n)).

3.  **Optimize Memory Usage for Large Files:**
    -   **Action (`load_large_dataset`):** Instead of `json.load()`, use a streaming JSON parser (e.g., `ijson`). This allows you to process the file item by item without loading the entire structure into memory.

### Quick Fixes

These are low-effort changes that immediately improve code quality and performance.

-   **Remove Dead Code:** Delete the `self.cache = []` line in `__init__` (Line 6) as it is never used.
-   **Optimize String Concatenation:** In the `concatenate_strings` function, replace the `+=` operator inside the loop with a single, efficient `''.join(list_of_strings)` call.
-   **Simplify Key Check:** In `load_large_dataset` (Line 16), replace the unnecessary `for key in item.keys():` loop with a direct and more readable check: `if 'id' in item:`.

### Code Health Score

**Score: Critical**

The codebase is assigned a "Critical" score due to the presence of a high-severity security vulnerability combined with fundamental performance issues that make it unreliable for its intended purpose of handling large datasets. The identified flaws pose a direct risk to the application's security and stability. The recommendations above are essential for making the code safe and functional.


--------------------------------------------------------------------------------


### 3.user_authentication.py

In [53]:
app = CodeReviewBuddy()
report, results = app.run_analysis()

Code Review Buddy initialized successfully!
Knowledge base loaded with comprehensive rules
AI agents ready for deep analysis



STEP 1: FILE UPLOAD AND PROCESSING

CODE UPLOAD OPTIONS
1. Upload Individual Files (specify paths)
2. Upload ZIP Archive
3. Clone GitHub Repository
4. Scan Directory



Enter your choice (1-4):  1



Enter file paths (one per line, empty line to finish):
Example: /path/to/your/file.py


File path:  C:\Users\saumy\Desktop\ML & AI Upgrade IIIT\GenAI\Semantic Spotter - Project\BYOP - Code Review Buddy\Test codes\user_authentication.py
File path:  


 Processed user_authentication.py (2.2KB)

Successfully processed 1 files

STEP 2: COMPREHENSIVE CODE ANALYSIS

Analyzing file 1/1: user_authentication.py
   Analyzing line metrics...
   Performing AST analysis...
   Running AI quality analysis...
   Performing security analysis...
   Analyzing performance...
   Analysis complete for user_authentication.py
Progress: 100.0% complete

STEP 3: GENERATING COMPREHENSIVE REPORT

STEP 4: ANALYSIS COMPLETE!



DETAILED ANALYSIS RESULTS

FILE: user_authentication.py

METADATA
  Language: python
  Size: 2.2 KB

METRICS
  Total Lines: 60
  Avg Line Length: 34.8
  Long Lines: 0
  Comment Lines: 8

STATIC ANALYSIS
  Complexity: 3
  Functions: 6
  Classes: 1
  Issues Found: 1

 CODE QUALITY
  Severity: CRITICAL
  Issues Found: 16
  Key Findings:
    1. **Location**: `create_tables` (line 14), `register_user` (line 23), `login` (line 35), `reset_password` (line 44).
    2. **Issue**: The code uses f-strings to insert user-provided data directly into SQL queries. This allows an attacker to inject malicious SQL commands. F
    3. **Recommendation**: **Never** use string formatting to build SQL queries. Use parameterized queries (placeholders), which the database driver will saf
   Detailed Report: code_review_reports\user_authentication.py_quality_20250930_002642.md

 SECURITY
  Severity: CRITICAL
  Issues Found: 12
  Key Findings:
    1. **Severity:** **Critical**
    2. **Location:** `register_use

In [55]:
summaries = get_file_summaries(results, app)


FILE SUMMARIES - MEDIUM DETAIL ANALYSIS

SUMMARY: user_authentication.py


Here is a summary of the code review for `user_authentication.py`.

This review of `user_authentication.py` highlights critical security and performance issues that require immediate attention. The current implementation is not safe for production use and exposes the application and its users to significant risk.

### Key Issues Found

1.  **SQL Injection Vulnerability (Critical)**: The application uses f-strings to insert user-provided data directly into SQL queries (e.g., lines 23, 35, 44). This is a classic SQL injection vulnerability, allowing an attacker to bypass authentication, steal data, or modify the database by crafting malicious input.
2.  **Insecure Password Hashing (Critical)**: Passwords are hashed using MD5 (e.g., lines 21, 33, 43), which is a cryptographically broken algorithm. MD5 is extremely fast, making it trivial for an attacker to crack hashed passwords using modern hardware or rainbow tables.
3.  **Poor Database Performance (Critical)**: User lookups are performed on the `username` column without a database index. This results in a full table scan (O(N) complexity) for every login attempt, which will cause severe performance degradation as the user base grows.

### Recommendations

#### Priority Fixes

1.  **Remediate SQL Injection with Parameterized Queries**: This is the highest priority. All SQL queries must be rewritten to use parameterized queries (placeholders). This separates the SQL command from the user data, preventing the database from interpreting user input as code.
    *   **Action**: Replace f-string formatting like `f"SELECT * FROM users WHERE username = '{username}'"` with the database driver's placeholder syntax, such as `cursor.execute("SELECT * FROM users WHERE username = ?", (username,))`.

2.  **Implement Strong Password Hashing**: Replace MD5 immediately with a modern, slow, and salted hashing algorithm designed for passwords.
    *   **Action**: Integrate a library like `passlib` and use **Bcrypt** or **Argon2**. This will securely hash new passwords and provide a safe way to verify existing ones during a transition period.

3.  **Optimize Database Lookups**: To ensure the system can scale, an index must be added to the column used for user lookups.
    *   **Action**: Add a `UNIQUE` index to the `username` column in the `users` table. This can be done with a single SQL command: `CREATE UNIQUE INDEX idx_username ON users (username);`. This will change lookup complexity from O(N) to a much faster O(log N).

### Quick Fixes

The most critical issues are also the most straightforward to fix.

*   **Parameterize All Queries**: Converting the 3-4 SQL queries in this file from f-strings to parameterized queries is a low-effort change that completely resolves the SQL injection vulnerability. This should be done immediately.
*   **Add the Database Index**: Executing the `CREATE UNIQUE INDEX` command is a one-time action that provides an immediate and significant performance improvement for all user login and registration operations.

### Code Health Score

**Score: Critical**

The code is assigned a "Critical" score due to the presence of severe, easily exploitable security vulnerabilities that fundamentally undermine its purpose. The combination of SQL injection and broken password hashing compromises user accounts and data integrity. Furthermore, the lack of database indexing demonstrates a critical performance flaw that makes the solution unscalable. These issues must be fully remediated before this code can be considered for any use.


--------------------------------------------------------------------------------


### 4.Test_codes1.zip
contains:
- data_processor.py
- user_authentication.py
- web_scraper.py

In [58]:
app = CodeReviewBuddy()
report, results = app.run_analysis()

Code Review Buddy initialized successfully!
Knowledge base loaded with comprehensive rules
AI agents ready for deep analysis



STEP 1: FILE UPLOAD AND PROCESSING

CODE UPLOAD OPTIONS
1. Upload Individual Files (specify paths)
2. Upload ZIP Archive
3. Clone GitHub Repository
4. Scan Directory



Enter your choice (1-4):  2

Enter path to ZIP file:  C:\Users\saumy\Desktop\ML & AI Upgrade IIIT\GenAI\Semantic Spotter - Project\BYOP - Code Review Buddy\Test codes\Test_codes1.zip


Found data_processor.py (2.9KB)
Found user_authentication.py (2.2KB)
Found web_scraper.py (3.3KB)

Successfully processed 3 files

STEP 2: COMPREHENSIVE CODE ANALYSIS

Analyzing file 1/3: data_processor.py
   Analyzing line metrics...
   Performing AST analysis...
   Running AI quality analysis...
   Performing security analysis...
   Analyzing performance...
   Analysis complete for data_processor.py
Progress: 33.3% complete

Analyzing file 2/3: user_authentication.py
   Analyzing line metrics...
   Performing AST analysis...
   Running AI quality analysis...
   Performing security analysis...
   Analyzing performance...
   Analysis complete for user_authentication.py
Progress: 66.7% complete

Analyzing file 3/3: web_scraper.py
   Analyzing line metrics...
   Performing AST analysis...
   Running AI quality analysis...
   Performing security analysis...
   Analyzing performance...
   Analysis complete for web_scraper.py
Progress: 100.0% complete

STEP 3: GENERATING COMPREHENSIVE REPOR


DETAILED ANALYSIS RESULTS

FILE: data_processor.py

METADATA
  Language: python
  Size: 2.9 KB

METRICS
  Total Lines: 93
  Avg Line Length: 30.2
  Long Lines: 0
  Comment Lines: 8

STATIC ANALYSIS
  Complexity: 20
  Functions: 7
  Classes: 1
  Issues Found: 0

 CODE QUALITY
  Severity: CRITICAL
  Issues Found: 16
  Key Findings:
    1. **L8: Unused Class Attribute**
    2. **Issue**: The `self.cache` attribute is initialized in the `__init__` method but is never used anywhere in the class. This is considered dead code an
    3. **Recommendation**: Remove the line `self.cache = []`.
   Detailed Report: code_review_reports\data_processor.py_quality_20250930_003057.md

 SECURITY
  Severity: CRITICAL
  Issues Found: 18
  Key Findings:
    1. **Vulnerability:** The `filepath` parameter in the `load_large_dataset` function is passed directly to the `open()` function without any validation or
    2. **Impact:** An attacker could read sensitive files such as configuration files (`/etc/hosts`

In [59]:
summaries = get_file_summaries(results, app)


FILE SUMMARIES - MEDIUM DETAIL ANALYSIS

SUMMARY: data_processor.py


Here is a summary of the code review for `data_processor.py`.

### Code Review Summary: `data_processor.py`

This review highlights critical security, performance, and quality issues that require immediate attention. The code contains a severe vulnerability and significant performance bottlenecks that will prevent it from scaling or being used safely in a production environment.

---

### Key Issues Found

-   **Path Traversal Vulnerability (Critical)**: The `load_large_dataset` function is vulnerable to path traversal. It directly uses a user-provided `filepath` to open a file, allowing an attacker to access sensitive system files (e.g., `../../etc/passwd`) outside the intended directory.
-   **Inefficient Memory Usage (Critical)**: The same `load_large_dataset` function loads the entire JSON file into memory with `json.load()`. This will cause the application to crash when processing files larger than the available RAM, making it unsuitable for its stated purpose.
-   **Quadratic (O(n²)) Algorithms (Critical)**: Multiple methods use highly inefficient algorithms. `find_duplicates` and `calculate_statistics` operate with O(n²) complexity, which will be extremely slow on large datasets. Similarly, `concatenate_strings` uses an inefficient string concatenation pattern.
-   **Unpythonic Logic (Major)**: The `filter_by_criteria` method (L37-46) uses a boolean flag (`match`) to manage state inside a loop. This pattern is verbose, error-prone, and less readable than more direct Pythonic alternatives.

### Recommendations

1.  **Prioritize Fixing the Path Traversal Vulnerability**: This is the most urgent issue.
    -   **Action**: In `load_large_dataset`, validate the user-provided `filepath`. Use `os.path.abspath` to resolve the path and ensure it resides within a predefined, trusted base directory. Reject any paths that attempt to traverse upwards (`../`).

2.  **Refactor Data Loading for Scalability**:
    -   **Action**: Modify `load_large_dataset` to stream data instead of loading it all at once. Use a library like `ijson` to parse the JSON file iteratively, yielding records one by one. This will fix the memory issue and allow the processing of very large files.

3.  **Optimize Core Algorithms**:
    -   **Action**: Replace the O(n²) algorithms with efficient, linear-time alternatives.
        -   In `find_duplicates`, use a Python `set` or `collections.Counter` to find duplicates in O(n) time.
        -   In `concatenate_strings`, replace the loop-based concatenation with `"".join(list_of_strings)`.
        -   In `calculate_statistics`, use built-in functions like `sorted()` or a library like NumPy for efficient statistical operations.

### Quick Fixes

-   **Remove Dead Code**: Delete the unused class attribute `self.cache = []` on L8.
-   **Simplify Filtering Logic**: Refactor the loop in `filter_by_criteria` (L37-46) to remove the `match` flag. A simple `if/else` or a generator expression with `any()` would be more readable and Pythonic.

### Code Health Score

**Score: Critical**

The codebase requires immediate and significant refactoring. The presence of a critical security vulnerability, combined with severe performance bottlenecks that undermine the core functionality of the module, makes it unfit for production use. The identified issues indicate a need for a thorough review of fundamental security and performance practices.


--------------------------------------------------------------------------------

SUMMARY: user_authentication.py


Here is a summary of the code review for `user_authentication.py`. The analysis has identified several critical issues that require immediate attention before this code can be considered for production use.

### Key Issues Found

-   **Critical: SQL Injection Vulnerabilities:** The application is highly vulnerable to SQL injection attacks. User-provided data is directly embedded into SQL queries using f-strings on lines 11, 22, 33, and 41. This allows an attacker to manipulate the database, bypass authentication (e.g., using `' OR '1'='1' --`), or exfiltrate sensitive data.

-   **Critical: Weak and Unsalted Password Hashing:** The current password hashing mechanism is insufficient. Without proper salting and a strong, modern algorithm (like bcrypt), stored passwords are susceptible to rainbow table and brute-force attacks, placing all user credentials at significant risk if the database is ever compromised.

-   **Critical: Missing Database Index on `username`:** The `login()` and `reset_password()` functions perform database lookups without an index on the `username` column. This results in a full table scan (O(N) complexity), causing performance to degrade linearly as the user base grows. This is a major scalability bottleneck.

### Recommendations

-   **Priority 1: Eradicate SQL Injection with Parameterized Queries:** Your highest priority is to refactor all database queries to use parameterized statements. This is the industry-standard defense against SQL injection. Instead of f-strings, use placeholders (e.g., `?` or `%s`) provided by your database driver to safely pass user input.
    -   **Action:** Modify the `execute` calls in `register_user()`, `login()`, and `reset_password()`. For example, change `f"SELECT * FROM users WHERE username = '{username}'"` to `"SELECT * FROM users WHERE username = ?"` and pass `(username,)` as a separate argument to the execute method.

-   **Priority 2: Implement Strong, Salted Password Hashing:** Replace the existing hashing function with a well-vetted library like `bcrypt` or `Argon2`. These libraries automatically handle salt generation and use computationally intensive algorithms that are resistant to modern attacks.
    -   **Action:** Integrate the `bcrypt` library. Use `bcrypt.hashpw()` to hash new passwords during registration and `bcrypt.checkpw()` to verify passwords during login.

### Quick Fixes

-   **Add a Database Index:** This is a low-effort, high-impact performance fix.
    -   **Action:** In your table creation logic, add the following SQL command after the `CREATE TABLE` statement: `CREATE INDEX idx_username ON users (username);`. This will dramatically speed up login and other user lookup operations.

-   **Address the Login Endpoint First:** The `login` function (line 22) is often the most targeted endpoint for SQL injection.
    -   **Action:** Immediately refactor the query in the `login` function to use parameterization. This mitigates the most immediate threat while you work on the other functions.

### Code Health Score

-   **Overall Assessment: Critical**
-   **Explanation:** The score is **Critical** due to the presence of severe, easily exploitable security vulnerabilities that compromise the entire system and its data. The combination of SQL injection flaws and weak password security makes the application fundamentally unsafe. Furthermore, the critical performance issue will prevent it from scaling effectively. These issues must be fully resolved before the code can be deployed.


--------------------------------------------------------------------------------

SUMMARY: web_scraper.py


Here is a summary of the code review for `web_scraper.py`. The analysis has identified several critical issues across security, performance, and code quality that require immediate attention.

### Key Issues Found

The script contains fundamental flaws that impact its security, performance, and reliability. The most significant issues are:

1.  **Critical Security Vulnerability (SSRF):** The `fetch_page` method directly uses URLs without validation. This exposes the application to Server-Side Request Forgery (SSRF), allowing a malicious actor to force the server to make requests to internal network resources or arbitrary external services.
2.  **Critical Performance Bottleneck:** The `scrape_multiple_pages` method fetches URLs sequentially in a loop. This design is highly inefficient and does not scale. For scraping even a moderate number of pages, the execution time will be unacceptably long, as the program waits for each network request to complete before starting the next.
3.  **Brittle and Inefficient HTML Parsing:** The script relies on the `re` module to parse HTML. Regular expressions are not suitable for parsing complex, nested structures like HTML and are prone to breaking with minor changes to a webpage's layout. This approach is both unreliable and slower than dedicated parsing libraries.
4.  **Overly Broad Exception Handling:** The use of generic `except Exception:` clauses throughout the code is a poor practice. It can mask unexpected errors, suppress important bug information, and make debugging significantly more difficult.

### Recommendations

To bring this code to a production-ready standard, the following high-impact changes are recommended as a priority:

*   **Priority 1: Mitigate SSRF Vulnerability**
    *   **Action:** Before making any request in `fetch_page`, validate the URL. Implement a strict allow-list for URL schemes, permitting only `http` and `https`. Use a library like `urllib.parse` to analyze the URL and ensure it conforms to expected patterns.

*   **Priority 2: Implement Concurrent Scraping**
    *   **Action:** Refactor `scrape_multiple_pages` to use a thread pool, such as `concurrent.futures.ThreadPoolExecutor`. This will enable fetching multiple pages in parallel, dramatically reducing the overall runtime. For 100 pages, this could decrease execution time from nearly a minute to just a few seconds.

*   **Priority 3: Adopt a Robust HTML Parser**
    *   **Action:** Replace all regular expression logic for data extraction with a dedicated HTML parsing library like `BeautifulSoup4` or `lxml`. This will make the scraper more resilient to HTML changes, improve parsing performance, and simplify the data extraction code.

### Quick Fixes

These low-effort changes can be implemented immediately to improve code quality and maintainability:

*   **Standardize Imports:** Move `import json` from line 80 (inside the `save_results` method) to the top of the file with the other imports. This adheres to PEP 8 standards and makes all dependencies visible at a glance.
*   **Refine Exception Handling:** Replace generic `except Exception:` blocks with more specific exceptions. For example, catch `requests.exceptions.RequestException` for network-related issues and `IOError` for file-writing problems.

### Code Health Score

*   **Overall Assessment:** **Critical**
*   **Explanation:** The codebase contains a high-severity security vulnerability, a major performance bottleneck that prevents scalability, and an unreliable core design (regex-based parsing). While the script may function for a narrow set of inputs, it is not secure, robust, or efficient enough for production use. The issues identified require immediate and substantial refactoring.


--------------------------------------------------------------------------------
